# Hybrid CNN-2 + CNN-5 + CNN-6 → mRMR → SVM

**Pipeline:**
1. Re-extract embeddings from `May11/Dataset` using all 3 CNNs **simultaneously** (same image order)
2. Fuse: 128-d + 256-d + 512-d = **896-d**
3. Select top-K features via **mRMR (MIQ)**
4. Tune & fit **RBF-SVM**
5. Report: accuracy, macro-F1, confusion matrix, classification report, inference time

> **Why re-extract?** Pre-saved embeddings from each CNN were extracted with shuffled DataLoaders — different image orders. Concatenating them would fuse features from different images. Re-extracting with `shuffle=False` guarantees alignment.

**Results saved to:** `Results/07-Hybrid_CNN_mRMR_PyTorch/`

In [1]:
# ================================================
# Cell 1 — Dependencies
# ================================================
# pymrmr requires Cython + pytz as build prerequisites.
# --no-build-isolation lets pip reuse the already-installed Cython
# instead of recreating an isolated build env that would miss it.
# umap-learn is optional — the UMAP projection in Cell 13 skips gracefully if absent.
import subprocess, sys
def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])
pip('Cython')
pip('pytz')
pip('--no-build-isolation', 'pymrmr')
pip('umap-learn')
print('Dependencies ready.')

Dependencies ready.


In [2]:
# ================================================
# Cell 2 — Imports, Paths & Config
# ================================================
# ── Standard library and scientific computing ────────────────────────────
import os, sys, time, json, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import pairwise_distances
import joblib
import umap

# ─── Paths ────────────────────────────────────────────────────────────────────
BASE = Path('/home/jenarththan/Desktop/FYP')

# May11 rebalanced dataset: 10,500 train / 1,500 val / 3,000 test (1,750/250/500 per class)
TRAIN_DIR = BASE / 'May11/Dataset/Training'
TEST_DIR  = BASE / 'May11/Dataset/Testing'

PTH_CNN2  = BASE / 'May11/Notebooks/Custom/CNN2_May11_Results/best_custom_2cnn_model.pth'
PTH_CNN5  = BASE / 'May11/Notebooks/Custom/CNN5_May11_Results/best_custom_5cnn_model.pth'
PTH_CNN6  = BASE / 'May11/Notebooks/Custom/CNN6_May11_Results/best_custom_6cnn_model.pth'

OUT_DIR   = BASE / 'May11/Notebooks/Custom/Hybrid_May11_Results'
EMB_DIR   = OUT_DIR / 'Embeddings'
PLOTS_DIR = OUT_DIR / 'Plots'
REPS_DIR  = OUT_DIR / 'Reports'
MDLS_DIR  = OUT_DIR / 'Models'
CKPT_DIR  = OUT_DIR / 'Checkpoints'
for d in [EMB_DIR, PLOTS_DIR, REPS_DIR, MDLS_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ─── Constants ────────────────────────────────────────────────────────────────
CLASS_NAMES = ['0', '100', '500', '1000', '1500', '2000']
NUM_CLASSES = 6
IMG_SIZE    = 224
BATCH_SIZE  = 32
SEED        = 42
WARMUP      = 20
N_RUNS      = 200

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
set_seed()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'Dataset: Train={TRAIN_DIR}')
print(f'         Test ={TEST_DIR}')
print(f'Output : {OUT_DIR}')


Device : cuda
Dataset: Train=/home/jenarththan/Desktop/FYP/May11/Dataset/Training
         Test =/home/jenarththan/Desktop/FYP/May11/Dataset/Testing
Output : /home/jenarththan/Desktop/FYP/May11/Notebooks/Custom/Hybrid_May11_Results


## CNN Model Definitions + Load
Models are loaded here and used both for embedding extraction and inference-time measurement.

In [3]:
# ================================================
# Cell 3 — CNN Model Definitions & Checkpoint Load
# ================================================
# Embedding dimensions: CNN-2=128-d, CNN-5=256-d, CNN-6=512-d. Fused: 896-d.
# Architectures reproduced verbatim so load_state_dict() needs no key remapping.
class Custom2CNN(nn.Module):
    def __init__(self, num_classes=6, p_drop=0.4):
        super().__init__()
        self.conv1 = nn.Conv2d(3,  32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool          = nn.MaxPool2d(2, 2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout       = nn.Dropout(p_drop)
        self.fc1           = nn.Linear(64, 128)
        self.fc2           = nn.Linear(128, num_classes)
    def forward(self, x, return_embedding=False):
        x   = self.pool(F.relu(self.conv1(x)))
        x   = self.pool(F.relu(self.conv2(x)))
        x   = self.adaptive_pool(x).view(x.size(0), -1)
        emb = F.relu(self.fc1(x))
        out = self.fc2(self.dropout(emb))
        return (out, emb) if return_embedding else out

class Custom5CNN(nn.Module):
    def __init__(self, num_classes=6, p_drop=0.5):
        super().__init__()
        self.conv1 = nn.Conv2d(3,   32,  kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32,  64,  kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64,  128, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.pool          = nn.MaxPool2d(2, 2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout       = nn.Dropout(p_drop)
        self.fc1           = nn.Linear(512, 256)
        self.fc2           = nn.Linear(256, num_classes)
    def forward(self, x, return_embedding=False):
        x   = self.pool(F.relu(self.conv1(x)))
        x   = self.pool(F.relu(self.conv2(x)))
        x   = self.pool(F.relu(self.conv3(x)))
        x   = self.pool(F.relu(self.conv4(x)))
        x   = self.pool(F.relu(self.conv5(x)))
        x   = self.adaptive_pool(x).view(x.size(0), -1)
        emb = F.relu(self.fc1(x))
        out = self.fc2(self.dropout(emb))
        return (out, emb) if return_embedding else out

class Custom6CNN(nn.Module):
    def __init__(self, num_classes=6, p_drop=0.5):
        super().__init__()
        self.conv1 = nn.Conv2d(3,    32,   kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32,   64,   kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64,   128,  kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(128,  256,  kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(256,  512,  kernel_size=3, padding=1)
        self.conv6 = nn.Conv2d(512,  1024, kernel_size=3, padding=1)
        self.pool          = nn.MaxPool2d(2, 2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout       = nn.Dropout(p_drop)
        self.fc1           = nn.Linear(1024, 512)
        self.fc2           = nn.Linear(512, num_classes)
    def forward(self, x, return_embedding=False):
        x   = self.pool(F.relu(self.conv1(x)))
        x   = self.pool(F.relu(self.conv2(x)))
        x   = self.pool(F.relu(self.conv3(x)))
        x   = self.pool(F.relu(self.conv4(x)))
        x   = self.pool(F.relu(self.conv5(x)))
        x   = F.relu(self.conv6(x))
        x   = self.adaptive_pool(x).view(x.size(0), -1)
        emb = F.relu(self.fc1(x))
        out = self.fc2(self.dropout(emb))
        return (out, emb) if return_embedding else out


def load_model(cls, pth_path, device):
    model = cls(num_classes=NUM_CLASSES)
    try:
        state = torch.load(str(pth_path), map_location=device, weights_only=True)
    except TypeError:
        state = torch.load(str(pth_path), map_location=device)
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    model.load_state_dict(state)
    return model.to(device).eval()

print('Loading models...')
cnn2 = load_model(Custom2CNN, PTH_CNN2, DEVICE)
cnn5 = load_model(Custom5CNN, PTH_CNN5, DEVICE)
cnn6 = load_model(Custom6CNN, PTH_CNN6, DEVICE)
print('All models loaded.')

rows = []
for name, model, pth, edim in [
    ('CNN-2', cnn2, PTH_CNN2, 128),
    ('CNN-5', cnn5, PTH_CNN5, 256),
    ('CNN-6', cnn6, PTH_CNN6, 512),
]:
    total = sum(p.numel() for p in model.parameters())
    rows.append({
        'Model'           : name,
        'Embedding Dim'   : edim,
        'Total Parameters': f'{total:,}',
        'Disk Size (MB)'  : round(os.path.getsize(str(pth)) / 1e6, 3),
    })
df_models = pd.DataFrame(rows)
print('\n' + df_models.to_string(index=False))
df_models.to_csv(REPS_DIR / 'model_parameter_summary.csv', index=False)

Loading models...
All models loaded.

Model  Embedding Dim Total Parameters  Disk Size (MB)
CNN-2            128           28,486           0.117
CNN-5            256        1,701,446           6.811
CNN-6            512        6,816,070          27.270


## Extract Embeddings (All 3 CNNs — Same Image Order)
All three CNNs process the **exact same batch** of images in sequence. `shuffle=False` guarantees row alignment. Results are cached so this only runs once.

In [4]:
# ================================================
# Cell 4 — Embedding Extraction  (shuffle=False alignment)
# ================================================
# Critical: shuffle=False guarantees all 3 CNNs process images in the same order.
# Row-aligned embeddings are needed for correct concatenation in Cell 6.
# Cached to disk so extraction only runs once.
CACHE_TR = EMB_DIR / 'aligned_fused_train.npz'
CACHE_TE = EMB_DIR / 'aligned_fused_test.npz'

if CACHE_TR.exists() and CACHE_TE.exists():
    print('Loading cached aligned embeddings...')
    tr_data = np.load(CACHE_TR)
    te_data = np.load(CACHE_TE)
    emb2_tr, emb5_tr, emb6_tr = tr_data['e2'], tr_data['e5'], tr_data['e6']
    emb2_te, emb5_te, emb6_te = te_data['e2'], te_data['e5'], te_data['e6']
    y_tr = tr_data['y']
    y_te = te_data['y']
else:
    # Match EXACTLY the transforms used during CNN training:
    # Resize + ToTensor only — NO normalization
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
    ])

    train_ds = datasets.ImageFolder(str(TRAIN_DIR), transform=transform)
    test_ds  = datasets.ImageFolder(str(TEST_DIR),  transform=transform)

    # Force numeric class order: 0,100,500,1000,1500,2000
    alpha_to_num = {old: CLASS_NAMES.index(cls)
                    for cls, old in train_ds.class_to_idx.items()}
    for ds in [train_ds, test_ds]:
        ds.targets = [alpha_to_num[t] for t in ds.targets]
        ds.samples = [(s, alpha_to_num[t]) for s, t in ds.samples]
        ds.class_to_idx = {n: i for i, n in enumerate(CLASS_NAMES)}

    # shuffle=False is critical — guarantees all 3 CNNs see the same image order
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, pin_memory=True)

    print(f'Train: {len(train_ds)} images  |  Test: {len(test_ds)} images')
    print(f'Class mapping: {train_ds.class_to_idx}')

    def extract_all(loader, desc):
        """Pass each batch through all 3 CNNs — guarantees same-image alignment."""
        e2_list, e5_list, e6_list, y_list = [], [], [], []
        for imgs, lbls in tqdm(loader, desc=desc):
            imgs = imgs.to(DEVICE)
            with torch.no_grad():
                _, e2 = cnn2(imgs, return_embedding=True)
                _, e5 = cnn5(imgs, return_embedding=True)
                _, e6 = cnn6(imgs, return_embedding=True)
            e2_list.append(e2.cpu().numpy())
            e5_list.append(e5.cpu().numpy())
            e6_list.append(e6.cpu().numpy())
            y_list.append(lbls.numpy())
        return (np.concatenate(e2_list), np.concatenate(e5_list),
                np.concatenate(e6_list), np.concatenate(y_list))

    print('\nExtracting train embeddings (all 3 CNNs per batch)...')
    emb2_tr, emb5_tr, emb6_tr, y_tr = extract_all(train_loader, 'Train')
    print('Extracting test embeddings...')
    emb2_te, emb5_te, emb6_te, y_te = extract_all(test_loader,  'Test')

    np.savez_compressed(CACHE_TR, e2=emb2_tr, e5=emb5_tr, e6=emb6_tr, y=y_tr)
    np.savez_compressed(CACHE_TE, e2=emb2_te, e5=emb5_te, e6=emb6_te, y=y_te)
    print('Aligned embeddings cached to Embeddings/')

print(f'\nShapes: CNN-2={emb2_tr.shape}  CNN-5={emb5_tr.shape}  CNN-6={emb6_tr.shape}')
print(f'Labels: train={y_tr.shape}  test={y_te.shape}')
print(f'Label order check (first 5): {[CLASS_NAMES[i] for i in y_tr[:5].tolist()]}')

Loading cached aligned embeddings...

Shapes: CNN-2=(10500, 128)  CNN-5=(10500, 256)  CNN-6=(10500, 512)
Labels: train=(10500,)  test=(3000,)
Label order check (first 5): ['0', '0', '0', '0', '0']


## Per-Image CNN Inference Time

In [5]:
# ================================================
# Cell 5 — CNN Extraction Inference Time
# ================================================
# WARMUP drains the CUDA kernel-launch queue — avoids cold-start bias in timing.
# N_RUNS=200 gives stable mean and std estimates.
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
for _ in range(WARMUP):
    with torch.no_grad():
        cnn2(dummy, return_embedding=True)
        cnn5(dummy, return_embedding=True)
        cnn6(dummy, return_embedding=True)

cnn_times = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    with torch.no_grad():
        cnn2(dummy, return_embedding=True)
        cnn5(dummy, return_embedding=True)
        cnn6(dummy, return_embedding=True)
    cnn_times.append((time.perf_counter() - t0) * 1000)

cnn_mean_ms = float(np.mean(cnn_times))
cnn_std_ms  = float(np.std(cnn_times))
print(f'CNN extraction per image ({N_RUNS} runs, device={DEVICE})')
print(f'  Mean: {cnn_mean_ms:.3f} ms  |  Std: {cnn_std_ms:.3f} ms')

CNN extraction per image (200 runs, device=cuda)
  Mean: 2.742 ms  |  Std: 0.063 ms


## Fuse Embeddings  (128 + 256 + 512 = 896-d)

In [6]:
# ================================================
# Cell 6 — Feature Fusion  (128 + 256 + 512 = 896-d)
# ================================================
# Concatenation along the feature axis — simplest fusion strategy.
# mRMR in Cell 7 discards redundant features across CNNs.
XtrF = np.concatenate([emb2_tr, emb5_tr, emb6_tr], axis=1)
XteF = np.concatenate([emb2_te, emb5_te, emb6_te], axis=1)
print(f'Fused train: {XtrF.shape}  |  Fused test: {XteF.shape}')

Fused train: (10500, 896)  |  Fused test: (3000, 896)


## mRMR Feature Selector

In [7]:
# ================================================
# Cell 7 — mRMR Feature Selector  (sklearn-compatible)
# ================================================
# Wraps pymrmr.mRMR (MIQ) as a sklearn transformer.
# min_var filter removes near-constant features before mRMR to avoid pymrmr
# producing incorrect column names. KBinsDiscretizer converts continuous
# embeddings to discrete bins required by mRMR.
import pymrmr

class MRMRSelector(BaseEstimator, TransformerMixin):
    def __init__(self, k=128, n_bins=10, strategy='quantile', min_var=1e-10):
        self.k=k; self.n_bins=n_bins; self.strategy=strategy; self.min_var=min_var
        self.selected_indices_=None; self.keep_idx_=None

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float32); y = np.asarray(y).astype(int)
        var = X.var(axis=0)
        self.keep_idx_ = np.where(var > self.min_var)[0]
        if self.keep_idx_.size == 0: self.keep_idx_ = np.arange(X.shape[1])
        Xk = X[:, self.keep_idx_]
        disc = KBinsDiscretizer(n_bins=self.n_bins, encode='ordinal', strategy=self.strategy)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            Xd = disc.fit_transform(Xk)
        if hasattr(Xd, 'toarray'): Xd = Xd.toarray()
        cols = [f'f{i}' for i in range(Xk.shape[1])]
        df   = pd.DataFrame(Xd, columns=cols); df.insert(0, 'class', y)
        k_eff = int(min(self.k, Xk.shape[1]))
        selected = pymrmr.mRMR(df, 'MIQ', k_eff)
        take = [n for n in selected if n.startswith('f')]
        idx_in_kept = []
        for n in take:
            try: idx_in_kept.append(int(n[1:]))
            except ValueError: pass
        if len(idx_in_kept) < k_eff:
            remaining = [i for i in range(Xk.shape[1]) if i not in idx_in_kept]
            idx_in_kept.extend(remaining[:(k_eff - len(idx_in_kept))])
        self.selected_indices_ = [int(self.keep_idx_[i]) for i in idx_in_kept]
        return self

    def transform(self, X):
        return np.asarray(X)[:, self.selected_indices_]

print('MRMRSelector ready.')

MRMRSelector ready.


## K* Selection — Mutual Information Elbow

In [8]:
# ================================================
# Cell 8 — Automatic K* Selection via MI Elbow
# ================================================
# Elbow = k where cumulative MI curve deviates most from its chord.
# Checkpointed so re-runs skip expensive MI computation.
def pick_k_elbow(X, y, k_min=32, k_max=512, seed=SEED):
    print('Computing mutual information per feature...')
    Xs     = StandardScaler().fit_transform(X).astype(np.float32)
    mi     = mutual_info_classif(Xs, y, random_state=seed, n_neighbors=3)
    order  = np.argsort(mi)[::-1]
    cumsum = np.cumsum(mi[order])
    xvals  = np.arange(1, len(cumsum) + 1)
    chord  = np.linspace(cumsum[0], cumsum[-1], num=len(cumsum))
    k_auto = int(xvals[np.argmax(np.abs(cumsum - chord))])
    k_auto = int(np.clip(k_auto, k_min, min(k_max, X.shape[1])))

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(xvals, cumsum, color='steelblue', lw=2, label='Cumulative MI')
    ax.plot(xvals, chord, color='gray', ls='--', alpha=0.7, label='Chord')
    ax.axvline(k_auto, color='red', ls=':', lw=2, label=f'Auto elbow  K={k_auto}')
    ax.fill_between(xvals, chord, cumsum, alpha=0.08, color='steelblue')
    ax.set_xlabel('Top-k features (by MI)', fontsize=12)
    ax.set_ylabel('Cumulative Mutual Information', fontsize=12)
    ax.set_title('Elbow Selection of K* (Mutual Information)', fontsize=13)
    ax.legend(fontsize=11); ax.grid(alpha=0.3, ls='--')
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'elbow_MI_cumsum.png', dpi=150)
    plt.show()
    print(f'Saved → Plots/elbow_MI_cumsum.png   Auto K* = {k_auto}')
    return k_auto

K_STAR_PATH = CKPT_DIR / 'k_star.json'
if K_STAR_PATH.exists():
    with open(K_STAR_PATH) as f: K_star = json.load(f)['K_star']
    print(f'Resumed K* = {K_star} from checkpoint.')
    pick_k_elbow(XtrF, y_tr)   # still show the plot
else:
    K_star = pick_k_elbow(XtrF, y_tr)
    with open(K_STAR_PATH, 'w') as f: json.dump({'K_star': K_star}, f)
print(f'Using K* = {K_star}')

Resumed K* = 256 from checkpoint.
Computing mutual information per feature...
Saved → Plots/elbow_MI_cumsum.png   Auto K* = 439
Using K* = 256


/tmp/ipykernel_5830/735168928.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# ================================================
# Cell 9 — Manual K* Override
# ================================================
# Changing K* invalidates all downstream checkpoints — wipe Checkpoints/
# to force clean re-run from Cell 10. Set K_OVERRIDE=None for auto elbow.
import shutil

# ── Manual K* override ───────────────────────────────────────────────────
# Set to None to use auto elbow, or set an integer to override
K_OVERRIDE = 256    # <── change this or set None for auto

if K_OVERRIDE is not None:
    K_star = K_OVERRIDE
    # Clear ALL checkpoint files including cv_C subfolder
    if CKPT_DIR.exists():
        shutil.rmtree(CKPT_DIR)
    CKPT_DIR.mkdir()
    with open(CKPT_DIR / 'k_star.json', 'w') as f:
        json.dump({'K_star': K_star}, f)
    print(f'K* overridden to {K_star} — all checkpoints cleared')
else:
    print(f'Using auto elbow K* = {K_star}')

K* overridden to 256 — all checkpoints cleared


## Hyperparameter Tuning  —  gamma + C
mRMR runs **once** on the full training set. The selected feature indices are then reused for all CV folds — no repeated mRMR calls.

In [ ]:
# ================================================
# Cell 10 — Hyperparameter Tuning  (mRMR + gamma* + C via 3-fold CV)
# ================================================
# Step 1: mRMR runs once; indices reused across CV folds (unsupervised selector).
# Step 2: gamma* = 1/(2*median_dist^2) — anchors RBF width to feature space scale.
# Step 3: C tuned by 3-fold stratified CV; all checkpointed for safe interrupt-resume.
import time, threading

def median_gamma(X, max_samples=500, seed=SEED):
    rng   = np.random.RandomState(seed)
    idx   = rng.choice(len(X), size=min(len(X), max_samples), replace=False)
    dists = pairwise_distances(X[idx], metric='euclidean', n_jobs=-1)
    iu    = np.triu_indices_from(dists, k=1)
    med   = np.median(dists[iu])
    if med <= 1e-12 or not np.isfinite(med): return 'scale'
    return float(1.0 / (2.0 * med ** 2))


MRMR_IDX_PATH = CKPT_DIR / f'mrmr_indices_K{K_star}.npy'
GAMMA_PATH    = CKPT_DIR / 'gamma_star.json'

# ── Step 1: mRMR once ────────────────────────────────────────────────────
print('=' * 60)
print('STEP 1 / 2  —  mRMR Feature Selection')
print(f'  K={K_star}  |  train_samples={len(XtrF)}  |  features={XtrF.shape[1]}')
print('=' * 60)

if MRMR_IDX_PATH.exists() and GAMMA_PATH.exists():
    mrmr_indices = np.load(MRMR_IDX_PATH)
    with open(GAMMA_PATH) as f: gamma_star = json.load(f)['gamma_star']
    print(f'[RESUMED] mRMR indices + gamma* = {gamma_star:.6g}')
else:
    # Background timer: prints elapsed time every 30 s while pymrmr runs
    _done = threading.Event()
    def _timer(start, done):
        while not done.wait(30):
            e = time.time() - start
            m, s = divmod(int(e), 60)
            print(f'  [mRMR] still running... {m}m {s}s elapsed', flush=True)
    _t0 = time.time()
    threading.Thread(target=_timer, args=(_t0, _done), daemon=True).start()

    sc_tmp  = StandardScaler().fit(XtrF)
    sel_tmp = MRMRSelector(k=K_star, n_bins=10, strategy='quantile')
    sel_tmp.fit(sc_tmp.transform(XtrF), y_tr)

    _done.set()
    m, s = divmod(int(time.time() - _t0), 60)
    print(f'[DONE] mRMR finished in {m}m {s}s')

    mrmr_indices = np.array(sel_tmp.selected_indices_)
    np.save(MRMR_IDX_PATH, mrmr_indices)
    X_sel      = sc_tmp.transform(XtrF)[:, mrmr_indices]
    gamma_star = median_gamma(X_sel)
    with open(GAMMA_PATH, 'w') as f: json.dump({'gamma_star': gamma_star}, f)
    print(f'  gamma* = {gamma_star:.6g}')

from_cnn2 = int(np.sum(mrmr_indices < 128))
from_cnn5 = int(np.sum((mrmr_indices >= 128) & (mrmr_indices < 384)))
from_cnn6 = int(np.sum(mrmr_indices >= 384))
print(f'\nFeature provenance (K={K_star}):')
print(f'  CNN-2 contributed: {from_cnn2} features ({from_cnn2/K_star*100:.1f}%)')
print(f'  CNN-5 contributed: {from_cnn5} features ({from_cnn5/K_star*100:.1f}%)')
print(f'  CNN-6 contributed: {from_cnn6} features ({from_cnn6/K_star*100:.1f}%)')

Xtr_sel = XtrF[:, mrmr_indices]
Xte_sel = XteF[:, mrmr_indices]

# ── Step 2: Tune C ────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('STEP 2 / 2  —  SVM C Hyperparameter Tuning  (3-fold CV)')
print(f'  C grid: (1, 10, 30, 100, 500)  |  15 SVM fits total')
print('=' * 60)

C_PATH = CKPT_DIR / 'C_star.json'
C_GRID = (1, 10, 30, 100, 500)

if C_PATH.exists():
    with open(C_PATH) as f: d = json.load(f)
    C_star, cv_f1 = d['C_star'], d['cv_f1']
    print(f'[RESUMED] C* = {C_star}  (cv macro-F1 = {cv_f1:.4f})')
else:
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    best_C, best_f1, cv_rows, fit_times = None, -1.0, [], []
    _cv_t0 = time.time()

    outer_bar = tqdm(C_GRID, desc='C candidates', position=0)
    for C in outer_bar:
        scores = []
        fold_bar = tqdm(
            enumerate(skf.split(Xtr_sel, y_tr), 1),
            total=3, desc=f'  C={C} folds', leave=False, position=1
        )
        for fold, (tr_i, va_i) in fold_bar:
            _ft = time.time()
            sc  = StandardScaler().fit(Xtr_sel[tr_i])
            clf = SVC(kernel='rbf', class_weight='balanced',
                      C=C, gamma=gamma_star, cache_size=2000)
            clf.fit(sc.transform(Xtr_sel[tr_i]), y_tr[tr_i])
            score = f1_score(y_tr[va_i],
                clf.predict(sc.transform(Xtr_sel[va_i])), average='macro')
            scores.append(score)
            fit_times.append(time.time() - _ft)
            fold_bar.set_postfix({
                'fold': fold,
                'f1': f'{score:.4f}',
                'fit': f'{fit_times[-1]:.0f}s'
            })

        mean_f1   = float(np.mean(scores))
        fits_done = len(fit_times)
        fits_left = 15 - fits_done
        avg_fit   = np.mean(fit_times)
        eta       = avg_fit * fits_left
        em, es    = divmod(int(eta), 60)
        outer_bar.set_postfix({
            'macro_f1': f'{mean_f1:.4f}',
            'avg_fit': f'{avg_fit:.0f}s',
            'ETA': f'{em}m{es}s'
        })
        print(f'  C={C:<5}  macro-F1={mean_f1:.4f}  '
              f'(avg fit so far: {avg_fit:.0f}s | ETA: {em}m {es}s)')
        cv_rows.append({'C': C, 'mean_macro_F1': round(mean_f1, 4)})
        if mean_f1 > best_f1: best_f1, best_C = mean_f1, C

    C_star, cv_f1 = best_C, best_f1
    with open(C_PATH, 'w') as f: json.dump({'C_star': C_star, 'cv_f1': cv_f1}, f)
    em, es = divmod(int(time.time() - _cv_t0), 60)
    print(f'\n[DONE] C tuning finished in {em}m {es}s')
    print('\nCV results:')
    print(pd.DataFrame(cv_rows).to_string(index=False))

print(f'\nK*={K_star}  |  C*={C_star}  |  gamma*={gamma_star:.6g}')


STEP 1 / 2  —  mRMR Feature Selection
  K=256  |  train_samples=10500  |  features=896


## Final Fit + Evaluate

In [ ]:
# ================================================
# Cell 11 — Final SVM Fit + Test Evaluation
# ================================================
# Fits SVM on ALL training data with tuned C* and gamma*.
# class_weight=balanced compensates for class imbalance; cache_size=2000 MB
# speeds up RBF kernel matrix for 10,500 training samples.
scaler_final = StandardScaler().fit(Xtr_sel)
clf_final    = SVC(kernel='rbf', class_weight='balanced',
                   C=C_star, gamma=gamma_star, cache_size=2000)
print('Fitting SVC...')
clf_final.fit(scaler_final.transform(Xtr_sel), y_tr)

print('Predicting on test set...')
y_pred = clf_final.predict(scaler_final.transform(Xte_sel))

acc  = accuracy_score(y_te, y_pred)
f1M  = f1_score(y_te, y_pred, average='macro')
f1m  = f1_score(y_te, y_pred, average='micro')
prec = precision_score(y_te, y_pred, average='macro', zero_division=0)
rec  = recall_score(y_te, y_pred,    average='macro', zero_division=0)

print(f'\n=== Test Results (K*={K_star}, C*={C_star}) ===')
print(f'  Accuracy       : {acc:.4f}')
print(f'  Macro-F1       : {f1M:.4f}')
print(f'  Micro-F1       : {f1m:.4f}')
print(f'  Macro-Precision: {prec:.4f}')
print(f'  Macro-Recall   : {rec:.4f}')

rep_str  = classification_report(y_te, y_pred, target_names=CLASS_NAMES, digits=4)
rep_dict = classification_report(y_te, y_pred, target_names=CLASS_NAMES,
                                  digits=4, output_dict=True)
print('\n' + rep_str)
with open(REPS_DIR / 'classification_report.txt', 'w') as f: f.write(rep_str)
pd.DataFrame(rep_dict).transpose().to_csv(REPS_DIR / 'classification_report.csv')
print('Saved → Reports/')

## Full Pipeline Inference Time Per Image

In [ ]:
# ================================================
# Cell 12 — Full Pipeline Inference Time Per Image
# ================================================
# Splits timing: cnn_ms = CNN only (t0->t1), full_ms = CNN+mRMR+SVM (t0->t2).
# Allows thesis to report SVM overhead separately from dominant CNN cost.
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
for _ in range(WARMUP):
    with torch.no_grad():
        _, e2 = cnn2(dummy, return_embedding=True)
        _, e5 = cnn5(dummy, return_embedding=True)
        _, e6 = cnn6(dummy, return_embedding=True)
    fused = np.concatenate([e2.cpu().numpy(), e5.cpu().numpy(),
                             e6.cpu().numpy()], axis=1)
    clf_final.predict(scaler_final.transform(fused[:, mrmr_indices]))

cnn_ms, full_ms = [], []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    with torch.no_grad():
        _, e2 = cnn2(dummy, return_embedding=True)
        _, e5 = cnn5(dummy, return_embedding=True)
        _, e6 = cnn6(dummy, return_embedding=True)
    t1 = time.perf_counter()
    fused = np.concatenate([e2.cpu().numpy(), e5.cpu().numpy(),
                             e6.cpu().numpy()], axis=1)
    clf_final.predict(scaler_final.transform(fused[:, mrmr_indices]))
    t2 = time.perf_counter()
    cnn_ms.append((t1 - t0) * 1000)
    full_ms.append((t2 - t0) * 1000)

cnn_mean_ms  = float(np.mean(cnn_ms));  cnn_std_ms  = float(np.std(cnn_ms))
full_mean_ms = float(np.mean(full_ms)); full_std_ms = float(np.std(full_ms))
print(f'Per-image inference time (device={DEVICE}, {N_RUNS} runs):')
print(f'  CNN extraction only : {cnn_mean_ms:.3f} ms ± {cnn_std_ms:.3f} ms')
print(f'  Full pipeline       : {full_mean_ms:.3f} ms ± {full_std_ms:.3f} ms')

timing_report = {
    'device': str(DEVICE), 'n_runs': N_RUNS,
    'cnn_extraction_mean_ms': round(cnn_mean_ms,  4),
    'cnn_extraction_std_ms' : round(cnn_std_ms,   4),
    'full_pipeline_mean_ms' : round(full_mean_ms, 4),
    'full_pipeline_std_ms'  : round(full_std_ms,  4),
}
with open(REPS_DIR / 'inference_time.json', 'w') as f:
    json.dump(timing_report, f, indent=2)
print('Saved → Reports/inference_time.json')

## Confusion Matrix, Per-Class Metrics & UMAP

In [ ]:
# ================================================
# Cell 13 — Visualisations  (Confusion Matrix, Per-Class Metrics, UMAP)
# ================================================
# UMAP projection skipped gracefully if umap-learn is not installed.
prec_c = precision_score(y_te, y_pred, average=None, labels=range(NUM_CLASSES), zero_division=0)
rec_c  = recall_score(y_te,  y_pred,   average=None, labels=range(NUM_CLASSES), zero_division=0)
f1_c   = f1_score(y_te,     y_pred,    average=None, labels=range(NUM_CLASSES), zero_division=0)

# Confusion matrix
cm   = confusion_matrix(y_te, y_pred, labels=list(range(NUM_CLASSES)))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(8, 7))
disp.plot(cmap='Blues', values_format='d', ax=ax)
ax.set_title(f'Confusion Matrix — Hybrid CNN-mRMR-SVM  K*={K_star}', fontsize=12)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'confusion_matrix.png', dpi=150)
plt.show()
print('Saved: Plots/confusion_matrix.png')

# Per-class bar chart
x, w = np.arange(NUM_CLASSES), 0.25
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w, prec_c, w, label='Precision', color='steelblue')
ax.bar(x,     rec_c,  w, label='Recall',    color='darkorange')
ax.bar(x + w, f1_c,   w, label='F1-Score',  color='seagreen')
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, fontsize=11)
ax.set_ylim(0, 1.05); ax.legend(fontsize=11)
ax.grid(axis='y', ls='--', alpha=0.5)
ax.set_title(f'Per-Class Metrics — Hybrid CNN-mRMR-SVM  K*={K_star}', fontsize=12)
ax.set_xlabel('Class (concentration µg/L)', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'per_class_metrics.png', dpi=150)
plt.show()
print('Saved: Plots/per_class_metrics.png')

# UMAP
try:
    Xt  = scaler_final.transform(Xtr_sel)
    X2d = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=SEED).fit_transform(Xt)
    fig, ax = plt.subplots(figsize=(7, 6))
    for i, name in enumerate(CLASS_NAMES):
        mask = (y_tr == i)
        ax.scatter(X2d[mask, 0], X2d[mask, 1], s=10, label=name, alpha=0.7)
    ax.legend(markerscale=2, fontsize=10, title='Class')
    ax.set_title(f'UMAP — mRMR Selected Features K*={K_star} (Train)', fontsize=12)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'umap_train.png', dpi=150)
    plt.show()
    print('Saved: Plots/umap_train.png')
except Exception as e:
    print(f'UMAP skipped: {e}')

## Save All Results

In [ ]:
# ================================================
# Cell 14 — Save All Artefacts
# ================================================
# Bundles scaler + clf + mrmr_indices into one joblib file for inference.
# Feature provenance CSV records per-CNN contribution to selected K* features.
joblib.dump({'scaler': scaler_final, 'clf': clf_final,
             'mrmr_indices': mrmr_indices},
            MDLS_DIR / 'svm_pipeline.joblib')
print('Pipeline saved → Models/svm_pipeline.joblib')

# Per-class CSV
pd.DataFrame([{
    'Class': cls, 'Precision': round(float(prec_c[i]), 4),
    'Recall': round(float(rec_c[i]), 4), 'F1': round(float(f1_c[i]), 4),
    'Support': int(np.sum(y_te == i))
} for i, cls in enumerate(CLASS_NAMES)]).to_csv(REPS_DIR / 'per_class_metrics.csv', index=False)

# Feature provenance CSV
pd.DataFrame([
    {'CNN': 'CNN-2', 'Features_Selected': from_cnn2, 'Pct': round(from_cnn2/K_star*100, 1)},
    {'CNN': 'CNN-5', 'Features_Selected': from_cnn5, 'Pct': round(from_cnn5/K_star*100, 1)},
    {'CNN': 'CNN-6', 'Features_Selected': from_cnn6, 'Pct': round(from_cnn6/K_star*100, 1)},
]).to_csv(REPS_DIR / 'feature_provenance.csv', index=False)

# Final summary
summary = {
    'dataset': 'May11_Dataset',
    'train_samples': int(len(y_tr)), 'test_samples': int(len(y_te)),
    'fused_dim': int(XtrF.shape[1]), 'K_star': int(K_star),
    'C_star': float(C_star), 'gamma_star': float(gamma_star) if not isinstance(gamma_star, str) else gamma_star,
    'cv_macro_f1': round(float(cv_f1), 4),
    'test_accuracy': round(acc, 4), 'test_macro_f1': round(f1M, 4),
    'test_micro_f1': round(f1m, 4), 'test_macro_precision': round(prec, 4),
    'test_macro_recall': round(rec, 4),
    'features_from_cnn2': from_cnn2, 'features_from_cnn5': from_cnn5,
    'features_from_cnn6': from_cnn6, **timing_report,
}
with open(REPS_DIR / 'final_summary.json', 'w') as f: json.dump(summary, f, indent=2)
pd.DataFrame([summary]).to_csv(REPS_DIR / 'final_summary.csv', index=False)

print('\n' + '='*60)
print('  FINAL SUMMARY')
print('='*60)
print(f'  Train / Test       : {len(y_tr)} / {len(y_te)}')
print(f'  Fused dim          : {XtrF.shape[1]}')
print(f'  K* / C* / gamma*   : {K_star} / {C_star} / {gamma_star:.6g}')
print(f'  Test Accuracy      : {acc:.4f}')
print(f'  Test Macro-F1      : {f1M:.4f}')
print(f'  CNN infer/img      : {cnn_mean_ms:.3f} ms ± {cnn_std_ms:.3f} ms')
print(f'  Full pipeline/img  : {full_mean_ms:.3f} ms ± {full_std_ms:.3f} ms')
print(f'  Feature provenance : CNN-2={from_cnn2} | CNN-5={from_cnn5} | CNN-6={from_cnn6}')
print('='*60)
print(f'\nAll results → {OUT_DIR}')